# Lesson 4: OOP & Modules

**Week 3 · Data Engineering Course**

---

**What you will learn:**
- Importing modules from the standard library
- Useful standard library tools: `os`, `datetime`, `collections`
- Writing reusable functions with docstrings
- Object-Oriented Programming: classes, `__init__`, `self`, methods
- Organising code: why modules matter in a pipeline

**This lesson focuses on patterns, not pandas.** Everything here scales to any Python project — ETL pipelines, APIs, scheduled jobs.

---

## 4.1 Importing Modules

Python ships with a large **standard library** — you do not need to `pip install` it. Import what you need.

In [ ]:
# Import the whole module
import os
import csv
import json

# Import specific names
from pathlib import Path
from datetime import datetime, date
from collections import defaultdict, Counter

print('Standard library imports OK')

In [ ]:
# os — operating system interface
cwd = os.getcwd()
print(f'Current directory: {cwd}')

# Environment variables (often used for secrets in pipelines)
path_env = os.environ.get('PATH', 'not found')
print(f'PATH starts with: {path_env[:60]}...')

# os.path — older path utilities (pathlib is preferred now, but you will see this in older code)
print(os.path.join('week3', 'data', 'raw'))  # cross-platform join

---

## 4.2 datetime — Working with Dates

Dates are everywhere in data engineering. The `datetime` module handles parsing, formatting, and arithmetic.

In [ ]:
from datetime import datetime, timedelta

# Get today's date
today = datetime.today()
print(f'Today: {today}')
print(f'Year: {today.year}, Month: {today.month}, Day: {today.day}')

# Format a date as a string
formatted = today.strftime('%Y-%m-%d')
print(f'Formatted: {formatted}')  # e.g. 2023-07-14

# Parse a date string into a datetime object
date_str = '2023-07-14'
parsed = datetime.strptime(date_str, '%Y-%m-%d')
print(f'Parsed: {parsed}')        # datetime(2023, 7, 14, 0, 0)

In [ ]:
# Handling multiple date formats — common in messy data
def parse_date(date_str):
    '''Parse a date string in multiple possible formats; return YYYY-MM-DD.'''
    formats = ['%Y-%m-%d', '%Y/%m/%d', '%d/%m/%Y', '%b %d %Y']
    for fmt in formats:
        try:
            return datetime.strptime(str(date_str).strip(), fmt).strftime('%Y-%m-%d')
        except ValueError:
            continue
    return None  # unrecognised format

test_dates = ['2023-01-05', '2023/07/14', '15/01/2023', 'Jan 20 2023', 'bad-date']
for d in test_dates:
    result = parse_date(d)
    print(f'  {d!r:20} -> {result}')

---

## 4.3 collections — Useful Container Types

In [ ]:
from collections import defaultdict, Counter

# Counter: count occurrences
categories = ['Electronics', 'Clothing', 'Electronics', 'Home & Kitchen', 'Electronics', 'Clothing']
counts = Counter(categories)
print(counts)
print(f'Most common: {counts.most_common(1)[0]}')

# defaultdict: dict that creates a default value for missing keys
revenue_by_region = defaultdict(float)
orders = [
    ('North', 1200.0),
    ('South', 75.0),
    ('North', 400.0),
    ('East', 150.0),
]
for region, amount in orders:
    revenue_by_region[region] += amount
print(dict(revenue_by_region))

---

## 4.4 Writing Reusable Functions

A well-structured pipeline puts each cleaning step into its own function. This makes the code testable, readable, and reusable across files.

In [ ]:
def clean_price(value):
    '''Remove currency symbols and convert to float. Returns None if unparseable.'''
    try:
        return float(str(value).replace('$', '').replace(',', '').strip())
    except (ValueError, TypeError):
        return None

def clean_category(value, typo_map=None):
    '''Normalise to title case and apply optional typo corrections.'''
    cleaned = str(value).strip().title()
    if typo_map:
        cleaned = typo_map.get(cleaned, cleaned)
    return cleaned

def parse_date(date_str):
    '''Parse common date formats to YYYY-MM-DD. Returns None if unrecognised.'''
    for fmt in ['%Y-%m-%d', '%Y/%m/%d', '%d/%m/%Y', '%b %d %Y']:
        try:
            return datetime.strptime(str(date_str).strip(), fmt).strftime('%Y-%m-%d')
        except ValueError:
            continue
    return None

# Test them
print(clean_price('$1,200.00'))      # 1200.0
print(clean_price('N/A'))            # None

typos = {'Electrnics': 'Electronics', 'Home & Kithen': 'Home & Kitchen'}
print(clean_category('ELECTRONICS', typos))     # Electronics
print(clean_category('Electrnics', typos))      # Electronics

print(parse_date('2023/07/14'))      # 2023-07-14
print(parse_date('bad'))             # None

---

## 4.5 Classes — Bundling Data and Behaviour

A **class** groups related data (attributes) and functions (methods) together. For data engineering, a class is useful when you have state that multiple functions need to share — for example, a pipeline that tracks statistics as it processes rows.

In [ ]:
# A minimal class
class Product:
    def __init__(self, name, category, price):
        self.name = name
        self.category = category
        self.price = price

    def describe(self):
        return f'{self.name} ({self.category}) — ${self.price:.2f}'

# Create instances
laptop = Product('Laptop', 'Electronics', 1200.00)
mouse = Product('Gaming Mouse', 'Electronics', 35.00)

print(laptop.describe())
print(mouse.describe())
print(f'Price difference: ${laptop.price - mouse.price:.2f}')

In [ ]:
# A more practical class: CleaningReport
# Tracks what the pipeline found and fixed during cleaning
class CleaningReport:
    def __init__(self, source_file):
        self.source_file = source_file
        self.total_rows = 0
        self.duplicates_removed = 0
        self.invalid_rows_dropped = 0
        self.missing_totals_filled = 0
        self.category_corrections = 0

    def record_read(self, row_count):
        self.total_rows = row_count

    def record_duplicate(self, count):
        self.duplicates_removed += count

    def record_invalid(self, count):
        self.invalid_rows_dropped += count

    def record_total_fill(self, count):
        self.missing_totals_filled += count

    def record_category_correction(self):
        self.category_corrections += 1

    def clean_rows(self):
        return self.total_rows - self.duplicates_removed - self.invalid_rows_dropped

    def summary(self):
        lines = [
            f'Source:              {self.source_file}',
            f'Total rows read:     {self.total_rows}',
            f'Duplicates removed:  {self.duplicates_removed}',
            f'Invalid rows dropped:{self.invalid_rows_dropped}',
            f'Missing totals fixed:{self.missing_totals_filled}',
            f'Category corrections:{self.category_corrections}',
            f'Clean rows output:   {self.clean_rows()}',
        ]
        return '\n'.join(lines)

# Simulate a cleaning run
report = CleaningReport('sales_q1.csv')
report.record_read(26)
report.record_duplicate(1)
report.record_invalid(1)
report.record_total_fill(2)
report.record_category_correction()
report.record_category_correction()

print(report.summary())

---

## 4.6 Organising Code into a Module

When a script grows, split it into modules — separate `.py` files, each with a clear responsibility.

A typical pipeline layout:

```
week3/
├── pipeline.py          <- orchestrates: read -> clean -> write
├── cleaners.py          <- clean_price(), clean_category(), parse_date()
├── validators.py        <- validate_row(), check_totals()
└── reporting.py         <- CleaningReport class
```

Then `pipeline.py` imports from the others:

```python
from cleaners import clean_price, clean_category, parse_date
from validators import validate_row
from reporting import CleaningReport
```

This is not required for the lab, but it is the pattern used in professional data pipelines.

In [ ]:
# Write a small helper module to a temporary file, then import it
from pathlib import Path

module_path = Path('temp_cleaners.py')
module_path.write_text('''"""Reusable cleaning helpers."""
from datetime import datetime

def clean_price(value):
    try:
        return float(str(value).replace("$", "").replace(",", "").strip())
    except (ValueError, TypeError):
        return None

def clean_category(value, typo_map=None):
    cleaned = str(value).strip().title()
    if typo_map:
        cleaned = typo_map.get(cleaned, cleaned)
    return cleaned

def parse_date(date_str):
    for fmt in ["%Y-%m-%d", "%Y/%m/%d", "%d/%m/%Y", "%b %d %Y"]:
        try:
            return datetime.strptime(str(date_str).strip(), fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return None
''', encoding='utf-8')

import importlib.util, sys

spec = importlib.util.spec_from_file_location('temp_cleaners', module_path)
cleaners = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cleaners)

# Use the imported functions
print(cleaners.clean_price('$350.00'))          # 350.0
print(cleaners.clean_category('electronics'))   # Electronics
print(cleaners.parse_date('2023/07/14'))         # 2023-07-14

module_path.unlink()  # clean up

---

## 4.7 Putting It Together — A Mini Pipeline

Here is a self-contained cleaning pipeline using everything from this week: functions, classes, pathlib, csv, exception handling.

In [ ]:
import csv
from pathlib import Path
from datetime import datetime

TYPO_MAP = {
    'Electrnics': 'Electronics',
    'Home & Kithen': 'Home & Kitchen'
}

def parse_date(s):
    for fmt in ['%Y-%m-%d', '%Y/%m/%d', '%d/%m/%Y', '%b %d %Y']:
        try:
            return datetime.strptime(str(s).strip(), fmt).strftime('%Y-%m-%d')
        except ValueError:
            pass
    return None

def clean_row(row, report):
    row['customer_name'] = row['customer_name'].strip()
    row['product'] = row['product'].strip()

    raw_cat = row['category'].strip().title()
    fixed = TYPO_MAP.get(raw_cat, raw_cat)
    if fixed != raw_cat:
        report.record_category_correction()
    row['category'] = fixed

    row['date'] = parse_date(row['date'])

    try:
        row['unit_price'] = float(str(row['unit_price']).replace('$', '').strip())
    except ValueError:
        row['unit_price'] = None

    try:
        row['quantity'] = int(float(row['quantity']))
    except ValueError:
        row['quantity'] = None

    if row['total'] == '' or row['total'] is None:
        if row['quantity'] and row['unit_price']:
            row['total'] = row['quantity'] * row['unit_price']
            report.record_total_fill(1)
    else:
        try:
            row['total'] = float(row['total'])
        except ValueError:
            row['total'] = None

    return row

def run_pipeline(input_path, output_path):
    report = CleaningReport(input_path.name)

    rows = []
    with open(input_path, 'r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(dict(row))
    report.record_read(len(rows))

    # Remove duplicates
    seen = set()
    unique = []
    for row in rows:
        if row['order_id'] in seen:
            report.record_duplicate(1)
        else:
            seen.add(row['order_id'])
            unique.append(row)

    # Clean each row
    cleaned = [clean_row(r, report) for r in unique]

    # Drop invalid rows
    valid = []
    for row in cleaned:
        if row['quantity'] is None or row['quantity'] <= 0:
            report.record_invalid(1)
        else:
            valid.append(row)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(valid[0].keys()))
        writer.writeheader()
        writer.writerows(valid)

    return report

input_path = Path('../data/raw/sales_q1.csv')
output_path = Path('../data/clean/sales_q1_pipeline.csv')

report = run_pipeline(input_path, output_path)
print(report.summary())

---

## 4.8 Key Takeaways

1. The standard library is large — `os`, `datetime`, `csv`, `json`, `collections`, `pathlib` cover most pipeline needs without external packages.
2. `datetime.strptime(s, fmt)` parses a date string. `datetime.strftime(fmt)` formats it back. Loop over multiple formats to handle messy data.
3. `Counter` counts values in one line. `defaultdict` removes the `if key not in dict` boilerplate.
4. A function should do **one thing**. Give it a clear name, a one-line docstring, and a return value — not a `print` statement.
5. A **class** combines state (attributes set in `__init__`) with behaviour (methods). Use one when multiple functions need to share the same data over time.
6. Split large scripts into modules (`.py` files). Import only what you need. This makes each piece independently testable.
7. The mini pipeline in Section 4.7 shows the canonical pattern: **read → deduplicate → clean each row → validate → write**. Every professional ETL tool is built on this idea.